# Stimulus-triggered responses

Loads one hf-visual session, puts the Bonsai streams (and Kilosort spikes, when available) on the NI-DAQ clock,
extracts the texture presentations, and looks at responses around each presentation:

1. **Behaviour** (no spike sorting needed): stimulus-triggered running speed, its trial-by-trial variation,
   and whether any texture changes running.
2. **Spikes** (needs `kilosort_dir`): rasters and peri-stimulus time histograms (PSTHs).

The work is done by groups of functions in `open_rigs_hf_visual`:

- `ephys`: SpikeGLX and Kilosort loading, and clock alignment from sync pulses (general).
- `spikes`: rasters and PSTHs around any event times (general).
- `traces`: averages and trial-by-trial views of continuous signals around any event times (general).
- `hf_visual`: this task's Bonsai streams, texture presentations and running speed (task specific).

All times below are seconds on the NI-DAQ clock.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import swc.aeon.io.api as aeon

from open_rigs_hf_visual import spikes as sp
from open_rigs_hf_visual import traces as tr
from open_rigs_hf_visual.hf_visual import load_session, running_speed, snap_to_edges, stimulus_table

## Session

`load_session` reads the Bonsai streams from `root` and the SpikeGLX run from `ephys_root`. It fits:

- **Arduino to NI-DAQ**, by matching the random-interval sync pulses (`last_sync_pulse_time` against `XA0`).
  The pulse offset between the two recordings is found automatically. The Arduino clock rate wanders (by about
  30 ms over a 25 min session), so the map interpolates between neighbouring pulses (about 1 ms accuracy).
- **Probe to NI-DAQ** (`align_probe=True`), from the 1 Hz SpikeGLX sync recorded by the SpikeGLX probe stream.
  This scans the whole probe file, so it is slow over the network.

`kilosort_dir` points at the spike sorting output. Its spike times must be on a clock that can be mapped to the
NI-DAQ: the SpikeGLX probe when `align_probe=True`.

In [ ]:
root = Path(r"Z:\ibn-vision\DATA\SUBJECTS\M26031\stimuli\20261008\M26031_20261008_1\ses-20261008_1_date-2026-10-08T15-12-29")
ephys_root = Path(r"Z:\ibn-vision\DATA\SUBJECTS\M26031\ephys\20261008\M26031_20261008_1\M26031_20261008_1_g2")
kilosort_dir = None  # e.g. Path(r"...\SpikeSorting_0\probe0\sorters\kilosort4")

session = load_session(root, ephys_root=ephys_root, kilosort_dir=kilosort_dir, align_probe=False)
print("Arduino -> NI-DAQ:", session.arduino_to_nidq)
print("Probe   -> NI-DAQ:", session.imec_to_nidq)
print("Task parameters:", session.task)

In [ ]:
# Sync check: deviation of each clock from a constant rate. A smooth curve is clock drift, which the piecewise
# Arduino map follows; isolated outliers would point to mismatched pulses.
maps = {"Arduino sync pulses": session.arduino_to_nidq, "Probe 1 Hz sync": session.imec_to_nidq}
maps = {name: m for name, m in maps.items() if m is not None}
fig, axes = plt.subplots(1, len(maps), figsize=(6 * len(maps), 3), squeeze=False)
for ax, (name, clock_map) in zip(axes[0], maps.items()):
    ax.plot(clock_map.residuals * 1e3, ".")
    ax.set(title=name, xlabel="pulse", ylabel="deviation from linear (ms)")
plt.tight_layout()

## Stimulus presentations

`stimulus_table` gives one row per texture presentation from the `Texture` log. Logged times use the latest
Arduino sample, so they are about 50 ms ahead of the real screen change, with ±9 ms jitter. The sync quad is on
while a texture is shown, and its photodiode is recorded on the NI-DAQ (10.6 kHz, NI-DAQ clock), so onsets are
moved to the next rising edge and offsets to the next falling edge (`snap_to_edges`).

`XA1` to `XA5` all carry screen signals; in M26031 `XA3` has exactly one edge per presentation. `threshold` is in
raw ADC units, between the dark (~680) and bright (~4100) levels. Without an NI-DAQ photodiode, use
`photodiode_edges(session.arduino["photodiode"])` instead (100 Hz, about 10 ms accuracy).

In [ ]:
blank = session.task.get("inter_presentation_texture", "blankStim_1")
textures = session.task.get("available_textures")
photodiode_channel, threshold = "XA3", 2400

logged = stimulus_table(session.texture, blank=blank, available_textures=textures)
rising = session.nidq.rising_edges(photodiode_channel, threshold=threshold)
falling = session.nidq.falling_edges(photodiode_channel, threshold=threshold)
onsets, matched = snap_to_edges(logged["onset"], rising, max_shift=0.15, lead=0.02)
offsets, _ = snap_to_edges(logged["offset"], falling, max_shift=0.15, lead=0.02)
print(f"{matched.sum()} of {len(matched)} onsets matched to a photodiode edge ({len(rising)} edges on {photodiode_channel})")

trials = stimulus_table(session.texture, blank=blank, available_textures=textures, onsets=onsets, offsets=offsets)
trials.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist((trials["onset"] - trials["onset_logged"])[matched] * 1e3, bins=30)
axes[0].set(xlabel="photodiode onset - logged onset (ms)", ylabel="presentations")
axes[1].hist(trials["duration"].dropna(), bins=30)
axes[1].set(xlabel="presentation duration (s)")
plt.tight_layout()
trials.groupby("texture").size()

### Alignment check

The photodiode itself, averaged around the corrected onsets, should step up sharply at 0 and down at the
presentation time. A ramp or a step away from 0 would mean the onsets are off.

In [ ]:
window = (-0.5, 1.0)  # seconds around each onset
presentation = session.task.get("presentation_time", 1.0)

photodiode = session.nidq.read(channels=[photodiode_channel], step=10)[photodiode_channel]  # ~1 kHz
aligned_photodiode = tr.align_trace(photodiode, trials["onset"], window, dt=0.001)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
tr.plot_mean(axes[0], aligned_photodiode)
axes[0].set(ylabel=f"{photodiode_channel} (ADC units)", title="Photodiode around corrected onsets")
tr.plot_mean(axes[1], tr.align_trace(photodiode, trials["onset_logged"], window, dt=0.001), color="C1")
axes[1].set(title="Photodiode around logged onsets")
plt.tight_layout()

## Running speed

`running_speed` differentiates the wheel encoder count. The wheel calibration is not in the rig schema, so speed is
in encoder counts/s; set `cm_per_count` (wheel circumference / counts per revolution) to get cm/s.

In [ ]:
cm_per_count = None  # e.g. np.pi * 20.0 / 1024 for a 20 cm diameter wheel and a 1024-count encoder
speed_units = "cm/s" if cm_per_count else "counts/s"
speed = running_speed(session.arduino, cm_per_count=cm_per_count, smooth=0.05)

fig, ax = plt.subplots(figsize=(12, 2.5))
ax.plot(aeon.to_seconds(speed.index), speed, lw=0.5, color="k")
ax.plot(trials["onset"], np.zeros(len(trials)), "|", color="C3", ms=6, label="stimulus onsets")
ax.set(xlabel="NI-DAQ time (s)", ylabel=f"speed ({speed_units})", title="Running speed over the session")
ax.legend(loc="upper right", frameon=False)
plt.tight_layout()

### Stimulus-triggered average

Mean speed (± SEM across trials) around every presentation, and separately for each texture family (the texture
name without its last `_n` index, e.g. `SN_42`).

In [ ]:
trials["family"] = trials["texture"].str.rsplit("_", n=1).str[0]
aligned_speed = tr.align_trace(speed, trials["onset"], window, dt=0.01)
by_family = tr.align_trace_by_condition(speed, trials["onset"], trials["family"], window, dt=0.01)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
tr.plot_mean(axes[0], aligned_speed, color="k")
axes[0].set(title=f"All presentations (n={aligned_speed.n_trials})", ylabel=f"speed ({speed_units})")
for family, aligned in by_family.items():
    tr.plot_mean(axes[1], aligned, label=f"{family} (n={aligned.n_trials})")
axes[1].set(title="By texture family")
axes[1].legend(fontsize="small", frameon=False)
for ax in axes:
    ax.axvspan(0, presentation, color="0.9", zorder=0)
plt.tight_layout()

### Trial-by-trial variation

Each row is one presentation in the order shown. Running comes in bouts, so most of the variation is between
trials rather than locked to the stimulus. Trials are classed as running when the speed in the 0.5 s before onset
is above `running_threshold`; this split is useful for the neural analyses too.

In [ ]:
running_threshold = 10.0  # counts/s (set in cm/s if cm_per_count is given)
trials["speed_pre"] = aligned_speed.trial_mean(window[0], 0.0)
trials["speed_stim"] = aligned_speed.trial_mean(0.0, presentation)
trials["speed_change"] = trials["speed_stim"] - trials["speed_pre"]
trials["running"] = trials["speed_pre"] > running_threshold
print(f"{trials['running'].mean():.0%} of trials start while running")

fig = plt.figure(figsize=(12, 7))
grid = fig.add_gridspec(2, 3, height_ratios=[3, 2])
ax = fig.add_subplot(grid[0, :2])
vmax = np.nanpercentile(aligned_speed.values, 99)
image = tr.plot_trials(ax, aligned_speed, vmin=0, vmax=vmax)
ax.axvline(0, color="w", lw=0.8)
ax.axvline(presentation, color="w", lw=0.8, ls="--")
fig.colorbar(image, ax=ax, label=f"speed ({speed_units})")
ax.set_title("Speed around each presentation, in trial order")

ax = fig.add_subplot(grid[0, 2], sharey=ax)
ax.plot(trials["speed_pre"], trials["trial"], ".", ms=2, color="k")
ax.set(xlabel=f"pre-onset speed ({speed_units})", title="Per trial")
ax.axvline(running_threshold, color="C3", lw=0.8)

ax = fig.add_subplot(grid[1, 0])
ax.scatter(trials["speed_pre"], trials["speed_stim"], s=4, c=trials["running"].map({True: "C3", False: "C0"}))
lim = [0, max(trials[["speed_pre", "speed_stim"]].max().max(), 1)]
ax.plot(lim, lim, color="0.5", lw=0.8)
ax.set(xlabel=f"before onset ({speed_units})", ylabel=f"during stimulus ({speed_units})", title="Speed before vs during")

ax = fig.add_subplot(grid[1, 1])
for state, color in ((False, "C0"), (True, "C3")):
    subset = trials["running"] == state
    tr.plot_mean(ax, tr.align_trace(speed, trials.loc[subset, "onset"], window), label=f"{'running' if state else 'stationary'} (n={subset.sum()})", color=color)
ax.axvspan(0, presentation, color="0.9", zorder=0)
ax.set(ylabel=f"speed ({speed_units})", title="Split by state at onset")
ax.legend(fontsize="small", frameon=False)

ax = fig.add_subplot(grid[1, 2])
ax.hist(trials["speed_change"], bins=60, color="0.3")
ax.set(xlabel=f"speed change ({speed_units})", ylabel="trials", title="During minus before")
plt.tight_layout()

### Does any texture change running?

Mean speed change (during minus before onset) per texture, ± SEM across its presentations. Bars that clearly
differ from zero, consistently across a family, would suggest the stimulus affects locomotion.

In [ ]:
summary = trials.groupby("texture").agg(
    family=("family", "first"),
    n=("trial", "size"),
    speed_pre=("speed_pre", "mean"),
    speed_stim=("speed_stim", "mean"),
    change=("speed_change", "mean"),
    change_sem=("speed_change", "sem"),
    running_fraction=("running", "mean"),
).sort_values(["family", "texture"])

fig, ax = plt.subplots(figsize=(12, 3.5))
colors = summary["family"].map({f: f"C{i}" for i, f in enumerate(summary["family"].unique())})
ax.bar(range(len(summary)), summary["change"], yerr=summary["change_sem"], color=colors, capsize=2)
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(range(len(summary)), summary.index, rotation=90, fontsize="small")
ax.set_ylabel(f"speed change ({speed_units})")
plt.tight_layout()
summary.round(2)

## Spikes

The cells below need spike sorting (`kilosort_dir` in the Session cell).

### Trials with probe data

The probe recording can stop before the session ends (in M26031 it goes flat after about 793 s). Trials outside
the span with probe data would look like silent responses, so they are dropped. `active_range` scans one channel
of the probe file.

In [ ]:
check_probe_range = False  # set True to scan the SpikeGLX probe file (slow over the network)
if check_probe_range and session.imec is not None:
    probe_start, probe_end = session.imec.active_range("AP0")
    to_nidq = session.imec_to_nidq or (lambda t: np.asarray(t))
    probe_start, probe_end = to_nidq([probe_start, probe_end])
    inside = (trials["onset"] + window[0] >= probe_start) & (trials["onset"] + window[1] <= probe_end)
    trials = trials[inside].reset_index(drop=True)
    print(f"probe data {probe_start:.1f} to {probe_end:.1f} s: keeping {len(trials)} trials")

## Units

In [ ]:
spikes = session.spikes
if spikes is None:
    raise RuntimeError("No spike sorting loaded: set kilosort_dir in the Session cell")
units = spikes.units
good = spikes.unit_ids("good")
print(f"{len(units)} clusters, {len(good)} labelled good")
units.head()

## Raster and PSTH of one unit

Trials are grouped and coloured by texture. The grey band is the presentation period.

In [ ]:
unit = good[0]
presentation = session.task.get("presentation_time", 1.0)

sp.plot_unit(
    spikes.spikes(unit),
    trials["onset"],
    trials["texture"],
    window=window,
    bin_size=0.02,
    smooth=0.02,
    shade=(0, presentation),
    title=f"unit {unit}",
);

### Effect of bin size

The same unit and texture with different bin widths, unsmoothed.

In [ ]:
texture = trials["texture"].iloc[0]
events = trials.loc[trials["texture"] == texture, "onset"]

fig, ax = plt.subplots(figsize=(8, 4))
for bin_size in (0.005, 0.01, 0.025, 0.05, 0.1):
    sp.plot_psth(ax, sp.psth(spikes.spikes(unit), events, window, bin_size), label=f"{bin_size * 1e3:g} ms", show_sem=False)
ax.axvspan(0, presentation, color="0.9", zorder=0)
ax.legend(title="bin size", frameon=False)
ax.set_title(f"unit {unit}, {texture}");

## Population

Mean PSTH of every good unit for each texture, normalised by each unit's peak across textures and sorted by
response latency to the first texture.

In [ ]:
bin_size = 0.025
trains = {u: spikes.spikes(u) for u in good}
names = sorted(trials["texture"].unique())
population = {
    name: sp.population_psth(trains, trials.loc[trials["texture"] == name, "onset"], window, bin_size, smooth=0.025)
    for name in names
}

baseline = np.mean([p.loc[:, p.columns < 0].mean(axis=1) for p in population.values()], axis=0)
peak = np.max([p.max(axis=1) for p in population.values()], axis=0)
scale = np.where(peak > baseline, peak - baseline, 1.0)
order = np.argsort(population[names[0]].loc[:, population[names[0]].columns > 0].to_numpy().argmax(axis=1))

fig, axes = plt.subplots(1, len(names), figsize=(4 * len(names), 6), sharey=True, squeeze=False)
for ax, name in zip(axes[0], names):
    normalised = (population[name].to_numpy() - baseline[:, None]) / scale[:, None]
    image = ax.imshow(normalised[order], aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1,
                      extent=[window[0], window[1], len(good) - 0.5, -0.5], interpolation="nearest")
    ax.axvline(0, color="k", lw=0.8)
    ax.axvline(presentation, color="k", lw=0.8, ls="--")
    ax.set(title=name, xlabel="time from onset (s)")
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
axes[0][0].set_ylabel("unit (sorted)")
fig.colorbar(image, ax=axes[0].tolist(), label="normalised rate")

### Response table

Mean rate during the presentation minus the mean rate in the 0.5 s before onset, per unit and texture.

In [ ]:
responses = pd.DataFrame(index=pd.Index(good, name="unit"), columns=names, dtype=float)
for u in good:
    for name in names:
        result = sp.psth(trains[u], trials.loc[trials["texture"] == name, "onset"], window, bin_size)
        responses.loc[u, name] = result.mean_rate(0.0, presentation).mean() - result.mean_rate(window[0], 0.0).mean()
responses.round(2)